# ω-Conotoxin ESMFold completion fallback

This notebook runs ESMFold v1 locally on a Colab GPU for the 17 structures that were missing when the public ESM Atlas REST endpoint stalled. It also folds one already-cached calibration sequence. The local importer compares that calibration result with the REST result before accepting any PDBs.

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Run both code cells in order. Installation and model download usually take a few minutes.
3. The second cell downloads `conotoxin_esmfold_colab.zip` automatically. Return that ZIP to the local project; do not rename or unpack its files manually.

The model and four-recycle setting match the official Meta ESMFold v1 defaults. This notebook is adapted from the [official ColabFold ESMFold notebook](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/ESMFold.ipynb).

In [ ]:
#@title Install ESMFold and download the v1 parameters
import os, subprocess
from pathlib import Path

MODEL_NAME = 'esmfold.model'
MODEL_URL = 'https://colabfold.steineggerlab.workers.dev/esm/esmfold.model'

if not Path('finished_install').is_file():
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2'], check=True)
    subprocess.run(['pip', 'install', '-q', 'omegaconf', 'pytorch_lightning',
                    'biopython', 'ml_collections', 'einops', 'modelcif'], check=True)
    subprocess.run(['pip', 'install', '-q',
                    'git+https://github.com/NVIDIA/dllogger.git'], check=True)
    subprocess.run(['pip', 'install', '-q',
                    'git+https://github.com/sokrypton/openfold.git'], check=True)
    subprocess.run(['pip', 'install', '-q',
                    'git+https://github.com/sokrypton/esm.git'], check=True)
    Path('finished_install').touch()

if not Path(MODEL_NAME).is_file():
    subprocess.run(['aria2c', '-q', '-c', '-x', '16', '-o', MODEL_NAME, MODEL_URL],
                   check=True)

size_gib = Path(MODEL_NAME).stat().st_size / 1024**3
print(f'Ready: {MODEL_NAME} ({size_gib:.2f} GiB)')

In [ ]:
#@title Fold the calibration and missing sequences, then download one ZIP
import csv, gc, hashlib, platform, shutil
from pathlib import Path

import torch
from google.colab import files

assert torch.cuda.is_available(), 'A GPU runtime is required (Runtime → Change runtime type → T4 GPU).'
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

jobs = [
    ('calibration', 'SA_strong_0031', 'CKGKGASCTRLSYDCCTGSCSSGKCG', 'CALIBRATION_SA_strong_SA_strong_0031_1b2d20c06460bc2d.pdb'),
    ('SA_strong', 'SA_strong_0372', 'CRRSGSSCSRTMYICCTGRCRSGKCG', 'SA_strong_SA_strong_0372_6ee6b78bd8379515.pdb'),
    ('SA_strong', 'SA_strong_0527', 'CKGKGAPCTRLSYDCCTGSCSSGRCG', 'SA_strong_SA_strong_0527_47a13fb789a1ba5e.pdb'),
    ('SA_strong', 'SA_strong_0589', 'CKRKGAPCGRTSYDCCSGSCSRGRCG', 'SA_strong_SA_strong_0589_28d988bbb71d6b03.pdb'),
    ('SA_strong', 'SA_strong_1209', 'CKGKGAKCSRLSYDCCSGSCSRGKCG', 'SA_strong_SA_strong_1209_e8b3cf7e166d9dbf.pdb'),
    ('SA_strong', 'SA_strong_1240', 'CKGKGAPCSRLMYDCCRGSCRSGKCG', 'SA_strong_SA_strong_1240_b7c85f18e6f17663.pdb'),
    ('SA_strong', 'SA_strong_1271', 'CKSTGSSCSPTSYNCCTGSCRPGKCG', 'SA_strong_SA_strong_1271_7d592013b63efb00.pdb'),
    ('SA_strong', 'SA_strong_1302', 'CKSAGKSCRRTAYDCCRGSCRSGKCG', 'SA_strong_SA_strong_1302_d83a458b7e10123b.pdb'),
    ('SA_strong', 'SA_strong_1333', 'CKSKGASCSKTMYDCCTGSCRRGRCY', 'SA_strong_SA_strong_1333_2b6987efbc538240.pdb'),
    ('SA_strong', 'SA_strong_1364', 'CKGKGASCRRTSYDCCTGSCRSGKCG', 'SA_strong_SA_strong_1364_ef898b19e18622e.pdb'),
    ('SA_strong', 'SA_strong_1395', 'CKPPGAPCRVSSYNCCSGSCKSKKCT', 'SA_strong_SA_strong_1395_dc5509fc03c771d4.pdb'),
    ('SA_strong', 'SA_strong_1488', 'CKSKGSKCRVTSYDCCTGSCRSGRCG', 'SA_strong_SA_strong_1488_66926692db1f17f.pdb'),
    ('SA_strong', 'SA_strong_1550', 'CKGAGAPCSRTAYNCCSGSCNSGRCG', 'SA_strong_SA_strong_1550_d085e28c3291c457.pdb'),
    ('SA_full', 'SA_full_1178', 'CKEPGAKCPVTSKDCCSGFCTLFFCM', 'SA_full_SA_full_1178_bbf0e69e966623f0.pdb'),
    ('SA_full', 'SA_full_1209', 'CLDGGTKCNRGNSQCCSGWCISLRCL', 'SA_full_SA_full_1209_dbd5f2225d423635.pdb'),
    ('SA_full', 'SA_full_1240', 'CSSGGSYCSSISYNCCTEFCAYLKCI', 'SA_full_SA_full_1240_2bdd2ab827429963.pdb'),
    ('SA_full', 'SA_full_1302', 'CKAEGEKCSSDSYDCCSGSCAYFKCE', 'SA_full_SA_full_1302_f6523b233cdfd4cc.pdb'),
    ('SA_full', 'SA_full_1550', 'CKPPGSFCRIFSLLCCKYYCSSKVCT', 'SA_full_SA_full_1550_16650992d88388c5.pdb'),
]

outdir = Path('conotoxin_esmfold_colab')
outdir.mkdir(exist_ok=True)

print('GPU:', torch.cuda.get_device_name(0))
print('Loading ESMFold v1...')
model = torch.load(MODEL_NAME, weights_only=False)
model = model.eval().cuda().requires_grad_(False)
model.set_chunk_size(128)

rows = []
for index, (source, name, sequence, filename) in enumerate(jobs, start=1):
    target = outdir / filename
    print(f'[{index:02d}/{len(jobs)}] {source}/{name} ({len(sequence)} aa)')
    with torch.no_grad():
        output = model.infer(sequence, num_recycles=4)
        pdb_text = model.output_to_pdb(output)[0]
        plddt = float(output['plddt'][0, ..., 1].mean().item())
    if pdb_text.count('\nATOM') < len(sequence) * 4:
        raise RuntimeError(f'Incomplete PDB generated for {name}')
    target.write_text(pdb_text)
    rows.append({
        'source': source, 'name': name, 'sequence': sequence,
        'sequence_sha256': hashlib.sha256(sequence.encode()).hexdigest(),
        'filename': filename, 'plddt': f'{plddt:.8f}',
    })
    del output, pdb_text
    gc.collect()
    torch.cuda.empty_cache()

with (outdir / 'manifest.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

with (outdir / 'RUN_METADATA.txt').open('w') as handle:
    handle.write(f'python={platform.python_version()}\n')
    handle.write(f'torch={torch.__version__}\n')
    handle.write(f'gpu={torch.cuda.get_device_name(0)}\n')
    handle.write('model=ESMFold v1\nnum_recycles=4\nchunk_size=128\nseed=0\n')

zip_path = shutil.make_archive('conotoxin_esmfold_colab', 'zip', root_dir='.', base_dir=outdir.name)
print(f'Completed {len(jobs)} predictions: {zip_path}')
files.download(zip_path)